<a href="https://colab.research.google.com/github/prometricas/William_Rondon/blob/main/CTD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cálculos inferenciales CTD – sección 4.4.3

Notebook para obtener las tablas estadísticas del análisis inferencial en la toma de decisiones en cuestiones sociocientíficas. No genera gráficos; solo produce tablas para normalidad, comparación entre grupos, cambios intragrupo y contraste de cambios.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from IPython.display import display

# Yo cargo el archivo validado; si no está en Colab, yo pido subirlo.
archivo = "data_Quimica_validado.xlsx"
try:
    ctd = pd.read_excel(archivo, sheet_name="CTD")
except FileNotFoundError:
    from google.colab import files
    subidos = files.upload()
    archivo = list(subidos.keys())[0]
    ctd = pd.read_excel(archivo, sheet_name="CTD")

# Yo verifico la cantidad de casos por grupo y momento.
print(pd.crosstab(ctd["grupo"], ctd["momento"]))
ctd.head()

Saving data_Quimica_validado.xlsx to data_Quimica_validado.xlsx
momento       Postest  Pretest
grupo                         
Control            23       23
Experimental       27       27


,momento,grupo,curso,id_estudiante,nombre_estudiante,sexo,edad,i01_csc1_estrategia_opcion_elegida,i02_csc1_estrategia_opciones_rechazadas,i03_csc1_ponderacion_criterios,i04_csc2_estrategia_opcion_elegida,i05_csc2_estrategia_opciones_rechazadas,i06_csc2_ponderacion_criterios,csc1_total,csc2_total,dim_estrategia_opcion_elegida,dim_estrategia_opciones_rechazadas,dim_ponderacion_criterios,ctd_total
0,Pretest,Control,1004,B_Est 1,Hanna Guarumo,F,16,2,0,0,1,0,0,2,1,3,0,0,3
1,Postest,Control,1004,B_Est 1,Hanna Guarumo,F,16,0,0,0,1,0,0,0,1,1,0,0,1
2,Pretest,Control,1004,B_Est 2,Ana Mileth Collo Valencia,F,16,0,1,0,1,0,0,1,1,1,1,0,2
3,Postest,Control,1004,B_Est 2,Ana Mileth Collo Valencia,F,16,1,0,0,1,0,0,1,1,2,0,0,2
4,Pretest,Control,1004,B_Est 3,Pineda Caro Liz,F,15,1,0,0,2,0,0,1,2,3,0,0,3


In [2]:
# Yo defino únicamente las variables necesarias para la sección 4.4.3.
variables = {
    "Estrategia para la opción elegida": "dim_estrategia_opcion_elegida",
    "Estrategia para las opciones rechazadas": "dim_estrategia_opciones_rechazadas",
    "Ponderación de criterios": "dim_ponderacion_criterios",
    "Puntaje total del CTD": "ctd_total"
}

grupos = ["Control", "Experimental"]
momentos = ["Pretest", "Postest"]

# Yo dejo funciones breves para redondear, resumir y calcular tamaños del efecto.
def r2(x):
    return np.nan if pd.isna(x) else round(float(x), 2)

def r3(x):
    return np.nan if pd.isna(x) else round(float(x), 3)

def p3(p):
    if pd.isna(p):
        return ""
    return "< 0.001" if p < 0.001 else f"{p:.3f}"

def resumen_mdrq(x):
    x = pd.Series(x).dropna().astype(float)
    mediana = x.median()
    q1 = x.quantile(0.25)
    q3 = x.quantile(0.75)
    return f"{mediana:.2f} [{q1:.2f}, {q3:.2f}]"

def rangos_medios(x, y):
    valores = np.r_[x, y]
    rangos = stats.rankdata(valores, method="average")
    return rangos[:len(x)].mean(), rangos[len(x):].mean()

def mann_whitney(x, y):
    x = np.asarray(x)
    y = np.asarray(y)
    n1, n2 = len(x), len(y)
    resultado = stats.mannwhitneyu(x, y, alternative="two-sided", method="asymptotic", use_continuity=True)
    u_x = resultado.statistic
    u_y = n1 * n2 - u_x
    u = min(u_x, u_y)
    rango_x, rango_y = rangos_medios(x, y)
    z_abs = 0 if resultado.pvalue >= 1 else abs(stats.norm.ppf(resultado.pvalue / 2))
    z = np.sign(rango_x - rango_y) * z_abs
    r = z_abs / np.sqrt(n1 + n2)
    return u, z, resultado.pvalue, r, rango_x, rango_y

def wilcoxon_pareado(pre, post):
    pre = np.asarray(pre)
    post = np.asarray(post)
    diferencia = post - pre
    if np.all(diferencia == 0):
        return 0, 0, 1, 0
    resultado = stats.wilcoxon(post, pre, zero_method="wilcox", correction=True, alternative="two-sided", method="approx")
    if hasattr(resultado, "zstatistic"):
        z = resultado.zstatistic
    else:
        z_abs = 0 if resultado.pvalue >= 1 else abs(stats.norm.ppf(resultado.pvalue / 2))
        z = np.sign(np.mean(diferencia)) * z_abs
    r = abs(z) / np.sqrt(len(pre))
    return resultado.statistic, z, resultado.pvalue, r

In [3]:
def normalidad(momento):
    filas = []
    datos = ctd[ctd["momento"] == momento]
    for nombre, col in variables.items():
        fila = {"Variable": nombre}
        for grupo in grupos:
            x = datos.loc[datos["grupo"] == grupo, col].dropna()
            w, p = stats.shapiro(x)
            fila[f"W {grupo.lower()}"] = r3(w)
            fila[f"p {grupo.lower()}"] = p3(p)
        filas.append(fila)
    return pd.DataFrame(filas)

def comparacion_intergrupal(momento):
    filas = []
    datos = ctd[ctd["momento"] == momento]
    for nombre, col in variables.items():
        control = datos.loc[datos["grupo"] == "Control", col].dropna().to_numpy()
        experimental = datos.loc[datos["grupo"] == "Experimental", col].dropna().to_numpy()
        u, z, p, r, rango_control, rango_experimental = mann_whitney(control, experimental)
        filas.append({
            "Variable": nombre,
            "Control Md [RIC]": resumen_mdrq(control),
            "Experimental Md [RIC]": resumen_mdrq(experimental),
            "U de Mann-Whitney": r2(u),
            "Z": r2(z),
            "p": p3(p),
            "r": r2(r)
        })
    return pd.DataFrame(filas)

def normalidad_diferencias():
    filas = []
    for grupo in grupos:
        datos = ctd[ctd["grupo"] == grupo]
        for nombre, col in variables.items():
            pre = datos.loc[datos["momento"] == "Pretest", ["id_estudiante", col]].rename(columns={col: "pretest"})
            post = datos.loc[datos["momento"] == "Postest", ["id_estudiante", col]].rename(columns={col: "postest"})
            pares = pre.merge(post, on="id_estudiante").dropna()
            diferencia = pares["postest"] - pares["pretest"]
            w, p = stats.shapiro(diferencia)
            filas.append({"Grupo": grupo, "Variable": nombre, "W": r3(w), "p": p3(p)})
    return pd.DataFrame(filas)

def cambios_intragrupo():
    filas = []
    for grupo in grupos:
        datos = ctd[ctd["grupo"] == grupo]
        for nombre, col in variables.items():
            pre = datos.loc[datos["momento"] == "Pretest", ["id_estudiante", col]].rename(columns={col: "pretest"})
            post = datos.loc[datos["momento"] == "Postest", ["id_estudiante", col]].rename(columns={col: "postest"})
            pares = pre.merge(post, on="id_estudiante").dropna()
            diferencia = pares["postest"] - pares["pretest"]
            w, z, p, r = wilcoxon_pareado(pares["pretest"], pares["postest"])
            filas.append({
                "Grupo": grupo,
                "Variable": nombre,
                "Pretest Md [RIC]": resumen_mdrq(pares["pretest"]),
                "Postest Md [RIC]": resumen_mdrq(pares["postest"]),
                "Cambio medio": r2(diferencia.mean()),
                "W": r2(w),
                "Z": r2(z),
                "p": p3(p),
                "r": r2(r)
            })
    return pd.DataFrame(filas)

def contraste_cambios():
    filas = []
    for nombre, col in variables.items():
        cambios = {}
        for grupo in grupos:
            datos = ctd[ctd["grupo"] == grupo]
            pre = datos.loc[datos["momento"] == "Pretest", ["id_estudiante", col]].rename(columns={col: "pretest"})
            post = datos.loc[datos["momento"] == "Postest", ["id_estudiante", col]].rename(columns={col: "postest"})
            pares = pre.merge(post, on="id_estudiante").dropna()
            cambios[grupo] = (pares["postest"] - pares["pretest"]).to_numpy()
        u, z, p, r, _, _ = mann_whitney(cambios["Experimental"], cambios["Control"])
        filas.append({
            "Variable": nombre,
            "Cambio control Md [RIC]": resumen_mdrq(cambios["Control"]),
            "Cambio experimental Md [RIC]": resumen_mdrq(cambios["Experimental"]),
            "Diferencia media de cambios": r2(np.mean(cambios["Experimental"]) - np.mean(cambios["Control"])),
            "U de Mann-Whitney": r2(u),
            "Z": r2(z),
            "p": p3(p),
            "r": r2(r)
        })
    return pd.DataFrame(filas)

In [4]:
# Yo genero las tablas principales de normalidad e inferencia intergrupal.
tabla_28_normalidad_pretest = normalidad("Pretest")
tabla_29_intergrupal_pretest = comparacion_intergrupal("Pretest")
tabla_30_normalidad_postest = normalidad("Postest")
tabla_31_intergrupal_postest = comparacion_intergrupal("Postest")

display(tabla_28_normalidad_pretest)
display(tabla_29_intergrupal_pretest)
display(tabla_30_normalidad_postest)
display(tabla_31_intergrupal_postest)

,Variable,W control,p control,W experimental,p experimental
0,Estrategia para la opción elegida,0.856,0.003,0.850,0.001
1,Estrategia para las opciones rechazadas,0.605,< 0.001,0.622,< 0.001
2,Ponderación de criterios,0.634,< 0.001,0.427,< 0.001
3,Puntaje total del CTD,0.909,0.040,0.876,0.004


,Variable,Control Md [RIC],Experimental Md [RIC],U de Mann-Whitney,Z,p,r
0,Estrategia para la opción elegida,"3.00 [2.00, 4.00]","3.00 [2.00, 4.00]",297.5,-0.26,0.797,0.04
1,Estrategia para las opciones rechazadas,"0.00 [0.00, 1.00]","0.00 [0.00, 1.00]",298.5,0.28,0.783,0.04
2,Ponderación de criterios,"0.00 [0.00, 1.00]","0.00 [0.00, 0.00]",258.0,1.40,0.160,0.20
3,Puntaje total del CTD,"4.00 [3.00, 4.00]","4.00 [3.00, 4.00]",289.5,0.42,0.675,0.06


,Variable,W control,p control,W experimental,p experimental
0,Estrategia para la opción elegida,0.902,0.027,0.726,< 0.001
1,Estrategia para las opciones rechazadas,0.783,< 0.001,0.750,< 0.001
2,Ponderación de criterios,0.592,< 0.001,0.750,< 0.001
3,Puntaje total del CTD,0.935,0.142,0.903,0.016


,Variable,Control Md [RIC],Experimental Md [RIC],U de Mann-Whitney,Z,p,r
0,Estrategia para la opción elegida,"2.00 [1.00, 3.00]","3.00 [3.00, 4.00]",133.0,-3.60,< 0.001,0.51
1,Estrategia para las opciones rechazadas,"1.00 [0.00, 1.00]","0.00 [0.00, 1.00]",284.5,0.55,0.585,0.08
2,Ponderación de criterios,"0.00 [0.00, 0.50]","0.00 [0.00, 1.00]",239.5,-1.60,0.109,0.23
3,Puntaje total del CTD,"3.00 [1.50, 4.00]","5.00 [4.00, 5.50]",164.5,-2.89,0.004,0.41


In [5]:
# Yo genero las tablas para los cambios pretest-postest y el contraste de cambios entre grupos.
tabla_32_normalidad_diferencias = normalidad_diferencias()
tabla_33_cambios_intragrupo = cambios_intragrupo()
tabla_34_contraste_cambios = contraste_cambios()

display(tabla_32_normalidad_diferencias)
display(tabla_33_cambios_intragrupo)
display(tabla_34_contraste_cambios)

,Grupo,Variable,W,p
0,Control,Estrategia para la opción elegida,0.920,0.066
1,Control,Estrategia para las opciones rechazadas,0.831,0.001
2,Control,Ponderación de criterios,0.816,< 0.001
3,Control,Puntaje total del CTD,0.901,0.027
4,Experimental,Estrategia para la opción elegida,0.881,0.005
5,Experimental,Estrategia para las opciones rechazadas,0.820,< 0.001
6,Experimental,Ponderación de criterios,0.783,< 0.001
7,Experimental,Puntaje total del CTD,0.952,0.239


,Grupo,Variable,Pretest Md [RIC],Postest Md [RIC],Cambio medio,W,Z,p,r
0,Control,Estrategia para la opción elegida,"3.00 [2.00, 4.00]","2.00 [1.00, 3.00]",-0.74,24.0,-2.55,0.011,0.53
1,Control,Estrategia para las opciones rechazadas,"0.00 [0.00, 1.00]","1.00 [0.00, 1.00]",0.35,9.0,-1.95,0.052,0.41
2,Control,Ponderación de criterios,"0.00 [0.00, 1.00]","0.00 [0.00, 0.50]",-0.09,18.0,-0.51,0.608,0.11
3,Control,Puntaje total del CTD,"4.00 [3.00, 4.00]","3.00 [1.50, 4.00]",-0.48,32.0,-1.30,0.195,0.27
4,Experimental,Estrategia para la opción elegida,"3.00 [2.00, 4.00]","3.00 [3.00, 4.00]",0.41,26.5,-1.99,0.046,0.38
5,Experimental,Estrategia para las opciones rechazadas,"0.00 [0.00, 1.00]","0.00 [0.00, 1.00]",0.26,15.0,-1.66,0.097,0.32
6,Experimental,Ponderación de criterios,"0.00 [0.00, 0.00]","0.00 [0.00, 1.00]",0.44,4.5,-2.61,0.009,0.50
7,Experimental,Puntaje total del CTD,"4.00 [3.00, 4.00]","5.00 [4.00, 5.50]",1.11,35.5,-3.17,0.002,0.61


,Variable,Cambio control Md [RIC],Cambio experimental Md [RIC],Diferencia media de cambios,U de Mann-Whitney,Z,p,r
0,Estrategia para la opción elegida,"-1.00 [-1.50, 0.00]","0.00 [0.00, 1.00]",1.15,150.0,3.24,0.001,0.46
1,Estrategia para las opciones rechazadas,"0.00 [0.00, 1.00]","0.00 [0.00, 1.00]",-0.09,291.0,-0.42,0.677,0.06
2,Ponderación de criterios,"0.00 [0.00, 0.00]","0.00 [0.00, 1.00]",0.53,210.0,2.21,0.027,0.31
3,Puntaje total del CTD,"0.00 [-1.00, 0.00]","1.00 [0.00, 2.00]",1.59,141.5,3.35,< 0.001,0.47


In [6]:
# Yo exporto todas las tablas a Excel para copiarlas en Word.
salida = "tablas_CTD_4_4_3.xlsx"
with pd.ExcelWriter(salida) as writer:
    tabla_28_normalidad_pretest.to_excel(writer, sheet_name="T28_normalidad_pre", index=False)
    tabla_29_intergrupal_pretest.to_excel(writer, sheet_name="T29_inter_pre", index=False)
    tabla_30_normalidad_postest.to_excel(writer, sheet_name="T30_normalidad_post", index=False)
    tabla_31_intergrupal_postest.to_excel(writer, sheet_name="T31_inter_post", index=False)
    tabla_32_normalidad_diferencias.to_excel(writer, sheet_name="T32_normalidad_dif", index=False)
    tabla_33_cambios_intragrupo.to_excel(writer, sheet_name="T33_cambios_intra", index=False)
    tabla_34_contraste_cambios.to_excel(writer, sheet_name="T34_contraste_cambios", index=False)

print(f"Archivo generado: {salida}")

# Yo intento descargar el Excel automáticamente cuando el notebook se ejecuta en Google Colab.
try:
    from google.colab import files
    files.download(salida)
except Exception:
    pass

Archivo generado: tablas_CTD_4_4_3.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>